In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, StackingRegressor
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor, XGBClassifier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Reload dataset (do not reuse old in-memory dataframe)
from pathlib import Path
_csv = Path('customer_prediction_system/predictor/data/dataset1.csv')
if not _csv.exists():
    _csv = Path('predictor/data/dataset1.csv')
if not _csv.exists():
    _csv = Path('../predictor/data/dataset1.csv')
df = pd.read_csv(_csv)
df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'], errors='coerce')
df['next_transaction_date'] = pd.to_datetime(df['next_transaction_date'], errors='coerce')
df = df.dropna(subset=['next_transaction_date'])

# Time-based train/test split (first 80% train, last 20% test)
df = df.sort_values('transaction_datetime').reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
train_df['_is_train'] = True
test_df['_is_train'] = False
df_combined = pd.concat([train_df, test_df], ignore_index=True)
df_combined = df_combined.sort_values(['customer_id', 'transaction_datetime']).reset_index(drop=True)

# Transaction gap (days between consecutive transactions per customer)
df_combined['_gap_days'] = df_combined.groupby('customer_id')['transaction_datetime'].diff().dt.total_seconds() / 86400
df_combined['_gap_days'] = df_combined['_gap_days'].fillna(0)

# Aggregations from train only (no leakage)
train_mask = df_combined['_is_train']
customer_avg_gap_map = df_combined.loc[train_mask].groupby('customer_id')['_gap_days'].mean()
vendor_avg_gap_map = df_combined.loc[train_mask].groupby('vendor')['_gap_days'].mean()
customer_txn_count_map = df_combined.loc[train_mask].groupby('customer_id').size()
vendor_txn_count_map = df_combined.loc[train_mask].groupby('vendor').size()
customer_vendor_txn_count_map = df_combined.loc[train_mask].groupby(['customer_id', 'vendor']).size()

df_combined['customer_avg_gap'] = df_combined['customer_id'].map(customer_avg_gap_map).fillna(0)
df_combined['vendor_avg_gap'] = df_combined['vendor'].map(vendor_avg_gap_map).fillna(0)
df_combined['customer_transaction_count'] = df_combined['customer_id'].map(customer_txn_count_map).fillna(0).astype(int)
df_combined['vendor_transaction_count'] = df_combined['vendor'].map(vendor_txn_count_map).fillna(0).astype(int)
cv_counts = df_combined.loc[train_mask].groupby(['customer_id', 'vendor']).size().reset_index(name='customer_vendor_transaction_count')
df_combined = df_combined.merge(cv_counts, on=['customer_id', 'vendor'], how='left')
df_combined['customer_vendor_transaction_count'] = df_combined['customer_vendor_transaction_count'].fillna(0).astype(int)

# Days since first transaction (per customer)
first_txn = df_combined.groupby('customer_id')['transaction_datetime'].transform('min')
df_combined['days_since_first_transaction'] = (df_combined['transaction_datetime'] - first_txn).dt.total_seconds() / 86400
df_combined['month'] = df_combined['transaction_datetime'].dt.month
df_combined['day_of_week'] = df_combined['transaction_datetime'].dt.dayofweek

# Drop helper columns and split back
train_df = df_combined[df_combined['_is_train']].drop(columns=['_is_train', '_gap_days']).reset_index(drop=True)
test_df = df_combined[~df_combined['_is_train']].drop(columns=['_is_train', '_gap_days']).reset_index(drop=True)

target = 'likelihood_prediction'
features = [
    'amount', 'vendor', 'transaction_type',
    'customer_transaction_count', 'vendor_transaction_count', 'customer_vendor_transaction_count',
    'customer_avg_gap', 'vendor_avg_gap',
    'days_since_first_transaction', 'month', 'day_of_week'
]
num_features = [
    'amount', 'customer_transaction_count', 'vendor_transaction_count', 'customer_vendor_transaction_count',
    'customer_avg_gap', 'vendor_avg_gap', 'days_since_first_transaction', 'month', 'day_of_week'
]
X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['vendor', 'transaction_type']),
    ('num', 'passthrough', num_features)
])

models = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(random_state=42),
    'XGBRegressor': XGBRegressor(random_state=42)
}

results = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R²': r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                Model       MAE      RMSE        R²
     LinearRegression 72.511118 85.421122 -0.249137
RandomForestRegressor 78.094952 98.654070 -0.666132
         XGBRegressor 78.235944 99.432253 -0.692521


In [3]:
from pathlib import Path
_csv = Path('customer_prediction_system/predictor/data/dataset1.csv')
if not _csv.exists():
    _csv = Path('predictor/data/dataset1.csv')
if not _csv.exists():
    _csv = Path('../predictor/data/dataset1.csv')
df = pd.read_csv(_csv)
df.head()

,first_name,last_name,customer_id,transaction_id,transaction_datetime,amount,vendor,transaction_type,next_transaction_date,likelihood_prediction
0,John,Miller,098a7cea-3c34-4b5a-89f6-2fa5c5388f0c,d969cf17-48b8-45ed-b129-3289aadfce9f,2024-01-17,470.46,Amazon,Debit,2024-05-20,124.0
1,John,Miller,098a7cea-3c34-4b5a-89f6-2fa5c5388f0c,64f65e02-6d1f-4ff6-af98-077e5a383c10,2024-05-20,344.23,Amazon,Credit,2024-10-02,135.0
2,John,Miller,098a7cea-3c34-4b5a-89f6-2fa5c5388f0c,278974ac-e953-4eaa-87bf-01e584bc7ed7,2024-10-02,163.82,Amazon,Credit,2024-12-06,65.0
3,John,Miller,098a7cea-3c34-4b5a-89f6-2fa5c5388f0c,8b55d14c-f3c3-4fb4-aaa5-b964b96977be,2024-12-06,169.62,Amazon,Credit,NaN,NaN
4,John,Miller,098a7cea-3c34-4b5a-89f6-2fa5c5388f0c,ee4ab9c6-8f41-4d7e-b063-7749c5998a6a,2024-11-11,126.07,American Airlines,Credit,2025-04-28,168.0


In [ ]:
print(df.shape)
df.info()

(5849, 10)
<class 'pandas.DataFrame'>
RangeIndex: 5849 entries, 0 to 5848
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   first_name             5849 non-null   str    
 1   last_name              5849 non-null   str    
 2   customer_id            5849 non-null   str    
 3   transaction_id         5849 non-null   str    
 4   transaction_datetime   5849 non-null   str    
 5   amount                 5849 non-null   float64
 6   vendor                 5849 non-null   str    
 7   transaction_type       5849 non-null   str    
 8   next_transaction_date  4021 non-null   str    
 9   likelihood_prediction  4021 non-null   float64
dtypes: float64(2), str(8)
memory usage: 457.1 KB


In [5]:
df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'], errors='coerce')
df['next_transaction_date'] = pd.to_datetime(df['next_transaction_date'], errors='coerce')
df = df.dropna(subset=['next_transaction_date'])

In [6]:
df['timing_category'] = np.where(df['likelihood_prediction'] <= 7, 0,
                                 np.where(df['likelihood_prediction'] <= 20, 1, 2))

In [ ]:
df = df.sort_values('transaction_datetime').reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print('Train shape:', train_df.shape)
print('Train dates:', train_df['transaction_datetime'].min(), 'to', train_df['transaction_datetime'].max())
print()
print('Test shape:', test_df.shape)
print('Test dates:', test_df['transaction_datetime'].min(), 'to', test_df['transaction_datetime'].max())

Train shape: (3216, 11)
Train dates: 2024-01-01 00:00:00 to 2024-10-15 00:00:00

Test shape: (805, 11)
Test dates: 2024-10-15 00:00:00 to 2025-11-15 00:00:00


In [8]:
target = 'likelihood_prediction'
features = ['amount', 'vendor', 'transaction_type']

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['vendor', 'transaction_type']),
    ('num', 'passthrough', ['amount'])
])

models = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(random_state=42),
    'XGBRegressor': XGBRegressor(random_state=42)
}

results = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R²': r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                Model       MAE      RMSE        R²
     LinearRegression 72.511118 85.421122 -0.249137
RandomForestRegressor 78.094952 98.654070 -0.666132
         XGBRegressor 78.235944 99.432253 -0.692521


In [ ]:
# Combine train/test, sort by customer and datetime for gap calculation
train_df = train_df.copy()
test_df = test_df.copy()
train_df['_is_train'] = True
test_df['_is_train'] = False
df_combined = pd.concat([train_df, test_df], ignore_index=True)
df_combined = df_combined.sort_values(['customer_id', 'transaction_datetime']).reset_index(drop=True)

# timing_category: 0 if ≤7, 1 if 8-20, 2 if >20
df_combined['timing_category'] = np.where(df_combined['likelihood_prediction'] <= 7, 0,
                                         np.where(df_combined['likelihood_prediction'] <= 20, 1, 2))

# Transaction gap (days between consecutive transactions per customer)
df_combined['transaction_gap'] = df_combined.groupby('customer_id')['transaction_datetime'].diff().dt.total_seconds() / 86400
df_combined['transaction_gap'] = df_combined['transaction_gap'].fillna(0)

# Compute aggregations from train only (no leakage)
train_mask = df_combined['_is_train']
customer_avg_gap_map = df_combined.loc[train_mask].groupby('customer_id')['transaction_gap'].mean()
vendor_avg_gap_map = df_combined.loc[train_mask].groupby('vendor')['transaction_gap'].mean()
customer_txn_count_map = df_combined.loc[train_mask].groupby('customer_id').size()
vendor_txn_count_map = df_combined.loc[train_mask].groupby('vendor').size()

df_combined['customer_avg_gap'] = df_combined['customer_id'].map(customer_avg_gap_map).fillna(0)
df_combined['vendor_avg_gap'] = df_combined['vendor'].map(vendor_avg_gap_map).fillna(0)
df_combined['customer_txn_count'] = df_combined['customer_id'].map(customer_txn_count_map).fillna(0).astype(int)
df_combined['vendor_txn_count'] = df_combined['vendor'].map(vendor_txn_count_map).fillna(0).astype(int)

# Split back
train_df = df_combined[df_combined['_is_train']].drop(columns=['_is_train']).reset_index(drop=True)
test_df = df_combined[~df_combined['_is_train']].drop(columns=['_is_train']).reset_index(drop=True)

# Retrain with new features
features_v2 = ['amount', 'vendor', 'transaction_type', 'transaction_gap', 'customer_avg_gap', 'vendor_avg_gap', 'customer_txn_count', 'vendor_txn_count']
X_train_v2 = train_df[features_v2]
X_test_v2 = test_df[features_v2]

preprocessor_v2 = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['vendor', 'transaction_type']),
    ('num', 'passthrough', ['amount', 'transaction_gap', 'customer_avg_gap', 'vendor_avg_gap', 'customer_txn_count', 'vendor_txn_count'])
])

results_v2 = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor_v2), ('model', model)])
    pipe.fit(X_train_v2, y_train)
    y_pred = pipe.predict(X_test_v2)
    results_v2.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R²': r2_score(y_test, y_pred)
    })

results_v2_df = pd.DataFrame(results_v2)
print(results_v2_df.to_string(index=False))

                Model       MAE      RMSE        R²
     LinearRegression 65.444049 77.974898 -0.040852
RandomForestRegressor 73.675118 87.706214 -0.316862
         XGBRegressor 77.863350 97.737866 -0.635329


In [ ]:
# Add month and day_of_week from transaction_datetime
train_df['month'] = train_df['transaction_datetime'].dt.month
train_df['day_of_week'] = train_df['transaction_datetime'].dt.dayofweek
test_df['month'] = test_df['transaction_datetime'].dt.month
test_df['day_of_week'] = test_df['transaction_datetime'].dt.dayofweek

features_v3 = ['amount', 'vendor', 'transaction_type', 'customer_txn_count', 'vendor_txn_count', 'month', 'day_of_week']
X_train_v3 = train_df[features_v3]
X_test_v3 = test_df[features_v3]

preprocessor_v3 = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['vendor', 'transaction_type']),
    ('num', 'passthrough', ['amount', 'customer_txn_count', 'vendor_txn_count', 'month', 'day_of_week'])
])

results_v3 = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor_v3), ('model', model)])
    pipe.fit(X_train_v3, y_train)
    y_pred = pipe.predict(X_test_v3)
    results_v3.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R²': r2_score(y_test, y_pred)
    })

results_v3_df = pd.DataFrame(results_v3)
print(results_v3_df.to_string(index=False))

                Model       MAE      RMSE        R²
     LinearRegression 65.697978 77.994376 -0.041372
RandomForestRegressor 71.744758 87.757711 -0.318409
         XGBRegressor 76.900912 98.969457 -0.676802


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

target_clf = 'timing_category'
y_train_clf = train_df[target_clf]
y_test_clf = test_df[target_clf]

print('Train timing_category value_counts:')
print(y_train_clf.value_counts().sort_index())
print()
print('Test timing_category value_counts:')
print(y_test_clf.value_counts().sort_index())
print()

classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'XGBClassifier': XGBClassifier(random_state=42)
}

for name, clf in classifiers.items():
    pipe = Pipeline([('preprocessor', preprocessor_v3), ('model', clf)])
    pipe.fit(X_train_v3, y_train_clf)
    y_pred_clf = pipe.predict(X_test_v3)
    print(f'{name}:')
    print(f'  Accuracy: {accuracy_score(y_test_clf, y_pred_clf):.4f}')
    print(f'  F1-score (macro): {f1_score(y_test_clf, y_pred_clf, average="macro"):.4f}')
    print(f'  Confusion matrix:\n{confusion_matrix(y_test_clf, y_pred_clf)}')
    print()

Train timing_category value_counts:
timing_category
0     246
1     342
2    2628
Name: count, dtype: int64

Test timing_category value_counts:
timing_category
0     50
1    105
2    650
Name: count, dtype: int64

RandomForestClassifier:
  Accuracy: 0.8025
  F1-score (macro): 0.2970
  Confusion matrix:
[[  0   1  49]
 [  0   0 105]
 [  1   3 646]]

XGBClassifier:
  Accuracy: 0.7988
  F1-score (macro): 0.2960
  Confusion matrix:
[[  0   0  50]
 [  0   0 105]
 [  2   5 643]]



In [12]:
# Stratified train/test split (for classifier comparison)
df_strat = df.copy()
df_strat['month'] = df_strat['transaction_datetime'].dt.month
df_strat['day_of_week'] = df_strat['transaction_datetime'].dt.dayofweek
df_strat['customer_txn_count'] = df_strat['customer_id'].map(df_strat.groupby('customer_id').size()).fillna(0).astype(int)
df_strat['vendor_txn_count'] = df_strat['vendor'].map(df_strat.groupby('vendor').size()).fillna(0).astype(int)

features_v3 = ['amount', 'vendor', 'transaction_type', 'customer_txn_count', 'vendor_txn_count', 'month', 'day_of_week']
X_strat = df_strat[features_v3]
y_strat = df_strat['timing_category']

X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X_strat, y_strat, test_size=0.2, random_state=42, stratify=y_strat
)

preprocessor_v3 = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['vendor', 'transaction_type']),
    ('num', 'passthrough', ['amount', 'customer_txn_count', 'vendor_txn_count', 'month', 'day_of_week'])
])

classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'XGBClassifier': XGBClassifier(random_state=42)
}

print('Stratified split - train/test value_counts:')
print('Train:', y_train_strat.value_counts().sort_index().to_dict())
print('Test:', y_test_strat.value_counts().sort_index().to_dict())
print()

for name, clf in classifiers.items():
    pipe = Pipeline([('preprocessor', preprocessor_v3), ('model', clf)])
    pipe.fit(X_train_strat, y_train_strat)
    y_pred_strat = pipe.predict(X_test_strat)
    print(f'{name}:')
    print(f'  Accuracy: {accuracy_score(y_test_strat, y_pred_strat):.4f}')
    print(f'  F1-score (macro): {f1_score(y_test_strat, y_pred_strat, average="macro"):.4f}')
    print(f'  Confusion matrix:\n{confusion_matrix(y_test_strat, y_pred_strat)}')
    print()

Stratified split - train/test value_counts:
Train: {0: 237, 1: 357, 2: 2622}
Test: {0: 59, 1: 90, 2: 656}

RandomForestClassifier:
  Accuracy: 0.8075
  F1-score (macro): 0.3188
  Confusion matrix:
[[  2   1  56]
 [  0   0  90]
 [  3   5 648]]

XGBClassifier:
  Accuracy: 0.8050
  F1-score (macro): 0.3370
  Confusion matrix:
[[  2   2  55]
 [  2   3  85]
 [  4   9 643]]

